# Lab 3.7 - Amazon SageMaker: Hyperparameter Tuning

**Educate edition.** Replaces `en_us/3_7-machinelearning.ipynb`.

## Objectives
* Create a hyperparameter tuning job
* Tune an XGBoost model
* Compare the tuned model against the baseline using performance metrics

**Prerequisite:** run Labs 3.4, 3.5 and 3.6.

> **Cost warning - this is the most expensive lab in the module.**
> A tuning job runs *many* training jobs. The original lab's default
> settings can run 10 or more. On Track A we cap it at **4 jobs, 2 in
> parallel**, which is enough to demonstrate the mechanism. Track B
> does the same search in-notebook for free.

## Lab configuration - CHOOSE YOUR TRACK

This notebook runs in one of two modes. Set the flag in the next cell.

| | `USE_MANAGED_SAGEMAKER = False` (**Track B**) | `USE_MANAGED_SAGEMAKER = True` (**Track A**) |
|---|---|---|
| Where training runs | Inside this notebook | A separate managed SageMaker job |
| Extra AWS cost | **$0** | A few cents per job |
| Needs S3 bucket | No | Yes |
| Needs IAM execution role with S3 access | No | Yes |
| Needs `ml.*` training quota | No | **Yes** |
| You learn | The ML concepts | The ML concepts **+ the SageMaker managed workflow** |

**If you are on a $1 budget, or a restricted sandbox account, use
Track B.** It produces the same model and the same numbers. Track A is
what the original AWS Academy lab does, and is worth showing if your
account and budget allow it.

In [ ]:
# ================= LAB CONFIGURATION =================
USE_MANAGED_SAGEMAKER = False    # <-- set True for the managed-job track

# Only used when USE_MANAGED_SAGEMAKER = True.
# These are the smallest instance types that SageMaker supports for each
# role, chosen to keep the cost down.
TRAIN_INSTANCE    = 'ml.m5.large'
ENDPOINT_INSTANCE = 'ml.t2.medium'
TRANSFORM_INSTANCE = 'ml.m5.large'
# =====================================================

import warnings; warnings.simplefilter('ignore')
import pandas as pd, numpy as np, os, json

print('Track:', 'A (managed SageMaker)' if USE_MANAGED_SAGEMAKER
      else 'B (in-notebook, no extra AWS cost)')

## Step 1 - What is a hyperparameter?

A **parameter** is learned from the data during training (the split
values inside each tree).

A **hyperparameter** is set *before* training and controls how learning
happens. XGBoost's main ones:

| Hyperparameter | Effect | Typical range |
|---|---|---|
| `max_depth` | How deep each tree grows. Deeper = more complex, more prone to overfitting | 3 - 10 |
| `eta` (learning rate) | How much each tree contributes. Lower = slower but more robust | 0.01 - 0.3 |
| `min_child_weight` | Minimum data in a leaf. Higher = more conservative | 1 - 10 |
| `gamma` | Minimum gain required to split. Higher = simpler trees | 0 - 5 |
| `subsample` | Fraction of rows sampled per tree | 0.5 - 1.0 |
| `num_round` | Number of trees | 50 - 500 |

Tuning searches this space for the combination that scores best on the
**validation** set. The test set stays untouched until the very end -
otherwise you are tuning against your own final exam.

In [ ]:
feature_cols = json.load(open('feature_cols.json'))
train = pd.read_csv('train.csv', header=None, names=['target'] + feature_cols)
validation = pd.read_csv('validation.csv', header=None, names=['target'] + feature_cols)
test = pd.read_csv('test.csv', header=None, names=['target'] + feature_cols)

baseline = json.load(open('baseline_metrics.json'))
print('Baseline from Lab 3.6:')
for k, v in baseline.items():
    print(f'  {k:10s} {v:.3f}')

### Track A setup check

The tuning job below reuses the S3 data and container image that lab 3.4
recorded in `sm_context.json`. If you ran lab 3.4 on Track B, that file
does not exist - this cell creates it, so the notebook still stands alone.

On Track B this cell does nothing.

In [ ]:
# Track A needs lab 3.4's managed-training context and its S3 uploads.
if not USE_MANAGED_SAGEMAKER:
    print('Track B selected - nothing to set up.')
else:
    # ---- Track A: make sure lab 3.4's managed-training context exists ----
    # sm_context.json is written by lab 3.4's Track A cell. If you ran lab 3.4
    # on Track B (the default) and then switched to Track A here, that file
    # will not exist. Rather than fail, recreate it now - the same way this
    # notebook retrains the Track B model above if it is missing.
    # Cost: one managed training job, ~4 minutes, ~$0.01.
    if os.path.exists('sm_context.json'):
        ctx = json.load(open('sm_context.json'))
        print('Loaded sm_context.json from lab 3.4')
        print('  training job:', ctx['training_job'])
    else:
        print('sm_context.json not found.')
        print('Lab 3.4 was not run on Track A, so there is no managed model yet.')
        print('Running that training job now (~4 min, ~$0.01)...')
        print()
        import sagemaker
        from sagemaker.inputs import TrainingInput

        session = sagemaker.Session()
        region  = session.boto_region_name
        bucket  = session.default_bucket()
        prefix  = 'mlfoundations/lab3'
        role    = sagemaker.get_execution_role()

        train_uri = session.upload_data('train.csv', bucket=bucket,
                                        key_prefix=f'{prefix}/train')
        val_uri   = session.upload_data('validation.csv', bucket=bucket,
                                        key_prefix=f'{prefix}/validation')
        image_uri = sagemaker.image_uris.retrieve('xgboost', region,
                                                  version='1.7-1')

        estimator = sagemaker.estimator.Estimator(
            image_uri=image_uri,
            role=role,
            instance_count=1,
            instance_type=TRAIN_INSTANCE,
            output_path=f's3://{bucket}/{prefix}/output',
            sagemaker_session=session,
            max_run=1200,                 # hard stop after 20 min - cost guard
        )
        estimator.set_hyperparameters(
            objective='binary:logistic',
            num_round=100,
            max_depth=5,
            eta=0.2,
            subsample=0.8,
            min_child_weight=6,
            gamma=4,
            early_stopping_rounds=10,
        )
        estimator.fit({
            'train':      TrainingInput(train_uri, content_type='csv'),
            'validation': TrainingInput(val_uri,   content_type='csv'),
        })

        ctx = {'bucket': bucket, 'prefix': prefix, 'region': region,
               'model_data': estimator.model_data,
               'training_job': estimator.latest_training_job.name,
               'image_uri': image_uri}
        with open('sm_context.json', 'w') as f:
            json.dump(ctx, f, indent=2)
        print()
        print('Saved sm_context.json - the training instance has already terminated.')

## Step 2 - Run the search

### Track B - tuning inside the notebook (free)

We use **randomised search with cross-validation**. Random search
samples combinations from the ranges rather than trying every one; with
a limited budget it reliably finds near-best settings faster than an
exhaustive grid.

`cv=5` means each candidate is scored on 5 different train/validation
folds and averaged. On a dataset this small that is far more
trustworthy than a single split.

We optimise for **ROC AUC** rather than accuracy, because the classes
are imbalanced.

In [ ]:
if not USE_MANAGED_SAGEMAKER:
    from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
    from xgboost import XGBClassifier
    import scipy.stats as st

    X = pd.concat([train, validation])[feature_cols]
    y = pd.concat([train, validation])['target']

    param_dist = {
        'max_depth':        [2, 3, 4, 5, 6, 8, 10],
        'learning_rate':    st.uniform(0.01, 0.29),
        'n_estimators':     [50, 100, 200, 300],
        'min_child_weight': [1, 2, 4, 6, 8],
        'gamma':            st.uniform(0, 5),
        'subsample':        st.uniform(0.5, 0.5),
        'colsample_bytree': st.uniform(0.5, 0.5),
    }

    search = RandomizedSearchCV(
        XGBClassifier(objective='binary:logistic', eval_metric='logloss',
                      random_state=42),
        param_distributions=param_dist,
        n_iter=40,
        scoring='roc_auc',
        cv=StratifiedKFold(5, shuffle=True, random_state=42),
        random_state=42,
        n_jobs=-1,
        verbose=1,
    )
    search.fit(X, y)

    print()
    print(f'Best cross-validated AUC: {search.best_score_:.4f}')
    print('Best hyperparameters:')
    for k, v in sorted(search.best_params_.items()):
        print(f'  {k:18s} {v}')

    best_model = search.best_estimator_
else:
    print('Track A selected - skip this cell.')

#### The 10 best combinations

Look at how *close together* the top scores are. That is typical: many
different hyperparameter settings perform about equally well, and the
gap between the best and the tenth-best is often smaller than the noise
in the data. This is a good antidote to over-interpreting a tuning
result.

In [ ]:
if not USE_MANAGED_SAGEMAKER:
    res = pd.DataFrame(search.cv_results_)
    cols = ['rank_test_score', 'mean_test_score', 'std_test_score',
            'param_max_depth', 'param_learning_rate', 'param_n_estimators',
            'param_min_child_weight', 'param_gamma']
    top = res.sort_values('rank_test_score')[cols].head(10)
    top['mean_test_score'] = top['mean_test_score'].round(4)
    top['std_test_score'] = top['std_test_score'].round(4)
    top['param_learning_rate'] = top['param_learning_rate'].astype(float).round(3)
    top['param_gamma'] = top['param_gamma'].astype(float).round(2)
    display(top.set_index('rank_test_score'))
else:
    print('Track A selected - skip this cell.')

### Track A - a managed SageMaker hyperparameter tuning job

`HyperparameterTuner` launches `max_jobs` separate training jobs,
`max_parallel_jobs` at a time, using Bayesian optimisation: each round
of jobs informs the next.

**We cap this at 4 jobs / 2 parallel deliberately.** Each job is a
separate billed `ml.m5.large` instance. Raising `max_jobs` raises the
cost proportionally.

In [ ]:
if USE_MANAGED_SAGEMAKER:
    import sagemaker
    from sagemaker.tuner import (HyperparameterTuner, IntegerParameter,
                                 ContinuousParameter)
    from sagemaker.inputs import TrainingInput

    ctx = json.load(open('sm_context.json'))
    session = sagemaker.Session()
    bucket, prefix, region = ctx['bucket'], ctx['prefix'], ctx['region']

    estimator = sagemaker.estimator.Estimator(
        image_uri=ctx['image_uri'],
        role=sagemaker.get_execution_role(),
        instance_count=1,
        instance_type=TRAIN_INSTANCE,
        output_path=f's3://{bucket}/{prefix}/tuning-output',
        sagemaker_session=session,
        max_run=1200,
    )
    estimator.set_hyperparameters(objective='binary:logistic',
                                  eval_metric='auc',   # emits validation:auc
                                  num_round=100)

    tuner = HyperparameterTuner(
        estimator,
        objective_metric_name='validation:auc',
        objective_type='Maximize',
        hyperparameter_ranges={
            'max_depth':        IntegerParameter(2, 8),
            'eta':              ContinuousParameter(0.05, 0.3),
            'min_child_weight': IntegerParameter(1, 8),
            'gamma':            ContinuousParameter(0, 5),
            'subsample':        ContinuousParameter(0.6, 1.0),
        },
        max_jobs=4,            # COST CONTROL - each job is a billed instance
        max_parallel_jobs=2,
    )

    tuner.fit({
        'train':      TrainingInput(f's3://{bucket}/{prefix}/train/train.csv',
                                    content_type='csv'),
        'validation': TrainingInput(f's3://{bucket}/{prefix}/validation/validation.csv',
                                    content_type='csv'),
    })
    tuner.wait()

    print('Best training job:', tuner.best_training_job())
    analytics = tuner.analytics().dataframe()
    display(analytics.sort_values('FinalObjectiveValue', ascending=False))
else:
    print('Track B selected - skip this cell.')

## Step 3 - Evaluate the tuned model on the test set

Now, and only now, we touch the test set. This is the honest
comparison: baseline versus tuned, both scored on data neither model
has ever seen.

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)

y_true = test['target'].values

if not USE_MANAGED_SAGEMAKER:
    y_prob_tuned = best_model.predict_proba(test[feature_cols])[:, 1]
else:
    tuned_predictor = tuner.deploy(initial_instance_count=1,
                                   instance_type=ENDPOINT_INSTANCE)
    from sagemaker.serializers import CSVSerializer
    tuned_predictor.serializer = CSVSerializer()
    payload = test[feature_cols].to_csv(header=False, index=False)
    raw = tuned_predictor.predict(payload).decode('utf-8')
    y_prob_tuned = np.array([float(v) for v in raw.strip().split('\n')])
    tuned_predictor.delete_endpoint(delete_endpoint_config=True)
    print('Tuned endpoint deleted.')

y_pred_tuned = (y_prob_tuned > 0.5).astype(int)

tuned_metrics = {
    'accuracy':  accuracy_score(y_true, y_pred_tuned),
    'precision': precision_score(y_true, y_pred_tuned, zero_division=0),
    'recall':    recall_score(y_true, y_pred_tuned, zero_division=0),
    'f1':        f1_score(y_true, y_pred_tuned, zero_division=0),
    'auc':       roc_auc_score(y_true, y_prob_tuned),
}
print(json.dumps({k: round(v, 4) for k, v in tuned_metrics.items()}, indent=2))

## Step 4 - Compare the two models

Read the `change` column carefully, and remember the caution from Lab
3.6: with only ~31 test records, a change of one or two percentage
points is **within noise**. A tuned model that scores slightly lower on
this test set has not necessarily got worse.

In [ ]:
cmp = pd.DataFrame({
    'baseline': baseline,
    'tuned': tuned_metrics,
})
cmp['change'] = cmp['tuned'] - cmp['baseline']
cmp = cmp.round(4)
display(cmp)

n = len(y_true)
print(f'\nTest set size: {n} records')
print(f'One record changing sides moves accuracy by {1/n:.1%}')

better = (cmp.loc['auc', 'change'] > 0)
print()
print('AUC is the most reliable comparison here (threshold-independent).')
print('Tuning improved AUC.' if better else
      'Tuning did not improve AUC on this test set - see the note below.')

In [ ]:
import matplotlib.pyplot as plt

ax = cmp[['baseline', 'tuned']].plot(
    kind='bar', figsize=(9, 4.5), rot=0,
    title='Baseline vs tuned model - test set')
ax.set_ylim(0, 1.05)
ax.set_ylabel('score')
for c in ax.containers:
    ax.bar_label(c, fmt='%.3f', fontsize=8)
plt.tight_layout(); plt.show()

### Why tuning sometimes does not help

This result is worth understanding rather than glossing over:

1. **The dataset is tiny.** 310 rows total, 31 in test. There is very
   little signal to squeeze out, and the measurement is noisy.
2. **The baseline hyperparameters were already sensible.** They came
   from the AWS lab and are close to good defaults for this problem.
3. **Tuning optimises validation score, not test score.** With small
   data, a model can be tuned to fit the validation folds slightly too
   well.

On larger, messier datasets - like the flight-delay challenge lab -
tuning typically produces a much clearer gain. The right lesson is not
"tuning always helps" but "tune, then verify honestly on held-out
data, and be sceptical of small differences."

## Step 5 - Final cost check

Confirm nothing is left running.

In [ ]:
try:
    import boto3
    sm = boto3.client('sagemaker')
    eps = sm.list_endpoints()['Endpoints']
    print('Endpoints running:', len(eps))
    for e in eps:
        print('  STILL BILLING:', e['EndpointName'], e['EndpointStatus'])
    jobs = sm.list_training_jobs(StatusEquals='InProgress')['TrainingJobSummaries']
    print('Training jobs in progress:', len(jobs))
    if not eps and not jobs:
        print()
        print('Clean. Only the notebook instance is billing - stop it now.')
except Exception as e:
    print('Could not check:', type(e).__name__, e)

## Conclusion

You have:
* Created a hyperparameter tuning job
* Tuned an XGBoost model
* Compared tuned against baseline using held-out test metrics
* Seen why small differences on a small test set should not be
  over-interpreted

## You have now finished Module 3

**Final cleanup - do this now:**
1. Confirm the endpoint list above is empty
2. In the SageMaker console, **Stop** `MyNotebook`
3. If you are completely done, **Delete** the notebook instance
4. Optionally empty the `sagemaker-<region>-<account-id>` S3 bucket

Optional next: `Flight_Delay-Student.ipynb` (Challenge Lab 3).